In [132]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt




Define the list of tickers


In [133]:
stocks = pd.read_excel(r"C:\Users\Tomás Gutiérrez\Desktop\Erasmus\CMG\Optimization input.xlsx")

In [134]:
tickers = stocks["Ticker"]
price_targets = pd.to_numeric(stocks["Price Target"], errors="coerce")
confidence = pd.to_numeric(stocks["Confidence in view"], errors="coerce")
current_prices = pd.to_numeric(stocks["Last Closing Price"], errors="coerce")

df = pd.DataFrame({
    "Ticker": stocks["Ticker"],
    "PriceTarget": pd.to_numeric(stocks["Price Target"], errors="coerce"),
    "Confidence": pd.to_numeric(stocks["Confidence in view"], errors="coerce"),
    "CurrentPrices": pd.to_numeric(stocks["Last Closing Price"], errors="coerce")
})

# Set index to ticker symbols for consistency
df = df.set_index("Ticker")


In [135]:
df

,PriceTarget,Confidence,CurrentPrices
Ticker,,,
JD,200.000,0.90,29.000
AMT,170.000,0.40,180.070
MSTR,300.000,0.30,161.580
ESIF.DE,17.000,0.60,14.764
IDR,60.000,0.85,40.470
PALL,210.000,0.73,174.990
ZETA,27.000,0.95,20.010
META,800.000,0.10,672.970
NRG,171.644,0.30,156.040


Dates

In [136]:
current_date = datetime.today()
start_date = current_date - timedelta(days = 10*365)

Download Adjusted Close prices (Close prices account for dividends)

In [ ]:

adj_close_df = pd.DataFrame()

data = yf.download(tickers)
adj_close_df[tickers] = data["Close"]

returns = adj_close_df.pct_change().dropna(how="all")




'\'\nadj_close_df = pd.DataFrame()\n\ndata = yf.download(tickers)\nadj_close_df[tickers] = data["Close"]\n\nreturns = adj_close_df.pct_change().dropna(how="all")\n\n'

In [ ]:
''''

# This is not needed in case of using live data

returns = pd.read_csv("returns.csv", parse_dates=["Date"])
returns = returns.set_index("Date")

# Make sure everything is numeric
returns = returns.apply(pd.to_numeric, errors="coerce")

''' 


In [139]:
print(returns)

                  JD       AMT      MSTR   ESIF.DE       IDR      PALL  \
Date                                                                     
2000-01-04       NaN -0.010661 -0.059952       NaN  0.000000       NaN   
2000-01-05       NaN  0.030173  0.063776       NaN  0.000000       NaN   
2000-01-06       NaN -0.010460 -0.039568       NaN  0.000000       NaN   
2000-01-07       NaN  0.054968  0.050562       NaN -0.504000       NaN   
2000-01-10       NaN  0.088176  0.194296       NaN  0.000000       NaN   
...              ...       ...       ...       ...       ...       ...   
2026-01-21  0.024313  0.002363  0.022343 -0.004117 -0.095207 -0.004542   
2026-01-22  0.031992 -0.006847 -0.017276  0.016538  0.110952  0.042306   
2026-01-23 -0.005333  0.010172  0.013231 -0.010710  0.002577  0.050196   
2026-01-26 -0.002011  0.003972 -0.015511  0.003563 -0.115895 -0.005251   
2026-01-27       NaN       NaN       NaN  0.011607       NaN       NaN   

                ZETA      META       

Calculate Covariance Matrix


In [140]:
years = len(returns) / 252
trading_days = 252
returns_annualized = returns.mean() * trading_days  
mu_base = returns_annualized
cov_matrix = returns.cov() * trading_days
corr_matrix = returns.corr() 

In [141]:
print(cov_matrix)

               JD       AMT      MSTR   ESIF.DE       IDR      PALL      ZETA  \
JD       0.227725  0.021514  0.072751  0.023048  0.007403  0.033248  0.072881   
AMT      0.021514  0.189815  0.063482  0.004120  0.009073  0.013611  0.019737   
MSTR     0.072751  0.063482  0.541400  0.028410  0.008701  0.032107  0.230133   
ESIF.DE  0.023048  0.004120  0.028410  0.034415  0.006169  0.013541  0.023889   
IDR      0.007403  0.009073  0.008701  0.006169  2.159817  0.026366  0.033762   
PALL     0.033248  0.013611  0.032107  0.013541  0.026366  0.117246  0.029145   
ZETA     0.072881  0.019737  0.230133  0.023889  0.033762  0.029145  0.522983   
META     0.055470  0.019583  0.067733  0.014927  0.009381  0.013966  0.112103   
NRG      0.030195  0.032735  0.045496  0.012516  0.022373  0.021821  0.050694   
PDD      0.226309  0.017195  0.110841  0.023942  0.030374  0.034638  0.110832   
GRAB     0.096807  0.012922  0.172503  0.014852  0.028861  0.017260  0.094874   

             META       NRG

Show Correlation matrix 

In [142]:
corr_matrix_rounded = corr_matrix.round(2)

# Display as a styled table with colors
corr_matrix_rounded.style.background_gradient(cmap='coolwarm').format("{:.2f}")



,JD,AMT,MSTR,ESIF.DE,IDR,PALL,ZETA,META,NRG,PDD,GRAB
JD,1.00,0.18,0.23,0.23,0.01,0.20,0.18,0.31,0.16,0.62,0.29
AMT,0.18,1.00,0.20,0.09,0.01,0.16,0.10,0.20,0.31,0.09,0.08
MSTR,0.23,0.20,1.00,0.16,0.01,0.15,0.35,0.27,0.20,0.19,0.30
ESIF.DE,0.23,0.09,0.16,1.00,0.04,0.18,0.18,0.18,0.17,0.18,0.13
IDR,0.01,0.01,0.01,0.04,1.00,0.06,0.06,0.02,0.05,0.05,0.06
PALL,0.20,0.16,0.15,0.18,0.06,1.00,0.10,0.10,0.17,0.12,0.07
ZETA,0.18,0.10,0.35,0.18,0.06,0.10,1.00,0.35,0.18,0.22,0.22
META,0.31,0.20,0.27,0.18,0.02,0.10,0.35,1.00,0.23,0.26,0.26
NRG,0.16,0.31,0.20,0.17,0.05,0.17,0.18,0.23,1.00,0.12,0.16
PDD,0.62,0.09,0.19,0.18,0.05,0.12,0.22,0.26,0.12,1.00,0.29



Blend Price Baseline + views


In [143]:

mu_view = np.log(df["PriceTarget"] / df["CurrentPrices"])

# Get tickers from df index (which is now ticker symbols)
tickers = df.index

# Reindex all Series to align with tickers
mu_base = mu_base.reindex(tickers)
confidence = df["Confidence"]  # Use confidence directly from df
mu_view = mu_view.reindex(tickers)

# Blend: final = base + confidence * (view - base)
mu_final = mu_base + confidence * (mu_view - mu_base)

###

In [144]:
# Calibrated version for the future
""""
years = len(returns) / 252

confidence = pd.Series(index=tickers, dtype=float)


for ticker in df.index:
    kappa = df.loc[ticker, "Uncertainty"]

    if np.isinf(kappa) or kappa > 999:   # commodities 
        confidence[ticker] = 0.0
    else:
        confidence[ticker] = years / (years + kappa)



# Blend historical mean with views
current_prices = returns.iloc[-1]

mu_view = pd.Series(index=df.index, dtype=float)
for ticker in df.index:
    mu_view[ticker] = np.log(df.loc[ticker, "PriceTarget"] / current_prices[ticker])

mu_final = mu_base.copy()
for ticker in df.index:
    mu_final[ticker] += confidence[ticker] * (mu_view[ticker] - mu_base[ticker])
    """


'"\nyears = len(returns) / 252\n\nconfidence = pd.Series(index=tickers, dtype=float)\n\n\nfor ticker in df.index:\n    kappa = df.loc[ticker, "Uncertainty"]\n\n    if np.isinf(kappa) or kappa > 999:   # commodities \n        confidence[ticker] = 0.0\n    else:\n        confidence[ticker] = years / (years + kappa)\n\n\n\n# Blend historical mean with views\ncurrent_prices = returns.iloc[-1]\n\nmu_view = pd.Series(index=df.index, dtype=float)\nfor ticker in df.index:\n    mu_view[ticker] = np.log(df.loc[ticker, "PriceTarget"] / current_prices[ticker])\n\nmu_final = mu_base.copy()\nfor ticker in df.index:\n    mu_final[ticker] += confidence[ticker] * (mu_view[ticker] - mu_base[ticker])\n    '

##

In [145]:
problem = mu_base + confidence * (mu_view - mu_base)
print(problem[problem.isna()])


Series([], dtype: float64)


Calculate Portfolio performance metrics

In [146]:
# risk free rate
rf = 0.02

def sigma(weights, cov_matrix):
    variance = weights.T @ cov_matrix @ weights
    return np.sqrt(variance)

def expected_return(weights, returns):
    mu = returns # Series of expected returns
    return np.dot(weights, mu)  # Portfolio expected return

def sharpe_ratio(weights, returns_annualized, cov_matrix, rf):
    return (expected_return(weights, returns_annualized)- rf) / sigma(weights, cov_matrix)



    

Define Function to minimize (negative sharpe ratio)

In [147]:


def neg_sharpe_ratio(weights, returns_annualized, cov_matrix, rf):
    return - sharpe_ratio(weights, returns_annualized, cov_matrix, rf)



Set Constraints

In [148]:
constraints = { "type": "eq", "fun" : lambda weights: np.sum(weights) - 1}
bounds = [(0.05, 0.20) for _ in range(len(tickers))]


Set Initial Weights

In [149]:
initial_weights=  np.array([1/ len(tickers)] * len(tickers))

Optimize the weights to maximize sharpe ratio

In [150]:
optimized = minimize(neg_sharpe_ratio, initial_weights, args=(mu_final, cov_matrix, rf), method= "SLSQP", constraints = constraints, bounds=bounds)

optimal_weights = optimized.x

In [151]:
optimal_weights_percent = np.round(optimal_weights * 100, 2)  # Convert to % and round


# Combine with tickers
portfolio_weights = pd.DataFrame({
    "Ticker": tickers,
    "Weight (%)": optimal_weights_percent
})

# Display table
print(portfolio_weights)

     Ticker  Weight (%)
0        JD       20.00
1       AMT        5.00
2      MSTR        5.00
3   ESIF.DE       20.00
4       IDR        5.00
5      PALL       14.28
6      ZETA        5.00
7      META        7.51
8       NRG        8.21
9       PDD        5.00
10     GRAB        5.00


Maximize M2

In [152]:
def M2(weights, returns_annualized, cov_matrix, rf=0.02, f=0.002, c=3):

    Rp = expected_return(weights, returns_annualized)
    Dt = Rp - rf
    
    # Use WEEKLY standard deviation (not annualized)
    # Convert annualized cov_matrix to weekly by dividing by 52
    cov_matrix_weekly = cov_matrix / 52
    sigma_p_weekly = sigma(weights, cov_matrix_weekly)
    sigma_b_weekly = 0.016

    # term_1: sigma_benchmark / sigma_p (weekly basis)
    # NOTE: Currently using sigma_p_weekly as placeholder for benchmark sigma
    # Should be replaced with actual benchmark sigma (e.g., equal-weight portfolio)
    # For now: term_1 = sigma_p_weekly / max(sigma_p_weekly, f)
    term_1 = sigma_b_weekly / max(sigma_p_weekly, f)
    
    # term_2 handles sign scaling
    term_2 = (1/c) * (-Dt/abs(Dt)) if Dt != 0 else 0

    multiplier = max(term_1, term_2)
    return rf + Dt * multiplier

def neg_M2(weights, returns_annualized, cov_matrix, rf=0.02, f=0.002, c=3):
    return - M2(weights, returns_annualized, cov_matrix, rf, f, c)



In [153]:
f = 0.002
c = 3

optimized_M2 = minimize(
    neg_M2,
    initial_weights,
    args=(mu_final, cov_matrix, rf, f, c),
    method="SLSQP",
    constraints=constraints,
    bounds=bounds
)

optimal_weights_M2 = optimized_M2.x
optimal_weights_percent_M2 = np.round(optimal_weights_M2*100, 2)

portfolio_weights_M2 = pd.DataFrame({
    "Ticker": tickers,
    "Weight (%)": optimal_weights_percent_M2
})

print(portfolio_weights_M2)


     Ticker  Weight (%)
0        JD       20.00
1       AMT        5.00
2      MSTR        5.00
3   ESIF.DE       20.00
4       IDR        5.00
5      PALL       14.28
6      ZETA        5.00
7      META        7.46
8       NRG        8.25
9       PDD        5.00
10     GRAB        5.00


In [156]:
# Sharpe ratio of the Sharpe-optimal portfolio
sharpe_value = sharpe_ratio(optimal_weights, mu_final, cov_matrix, rf)
print(f"Sharpe Ratio for Sharpe-optimal portfolio: {sharpe_value:.4f}")

# M² of the M2-optimal portfolio (already calculated)
M2_value = M2(optimal_weights_M2, mu_final, cov_matrix, rf, f, c)
print(f"M² for M2-optimal portfolio: {M2_value:.4f}")

Sharpe Ratio for Sharpe-optimal portfolio: 2.1025
M² for M2-optimal portfolio: 0.2626


In [157]:
# Create summary table
metrics_data = {
    "Metric": [
        "Expected Return",
        "Volatility",
        "Sharpe Ratio",
        "M²"
    ],
    "Sharpe-Optimal": [
        f"{sharpe_return:.4f}",
        f"{sharpe_volatility:.4f}",
        f"{sharpe_value:.4f}",
        f"{M2(optimal_weights, mu_final, cov_matrix, rf, f, c):.4f}"
    ],
    "M2-Optimal": [
        f"{M2_return:.4f}",
        f"{M2_volatility:.4f}",
        f"{sharpe_ratio(optimal_weights_M2, mu_final, cov_matrix, rf):.4f}",
        f"{M2_value:.4f}"
    ]
}

metrics_df = pd.DataFrame(metrics_data)
print("\n" + "="*70)
print("PORTFOLIO PERFORMANCE COMPARISON")
print("="*70)
print(metrics_df.to_string(index=False))
print("="*70)


PORTFOLIO PERFORMANCE COMPARISON
         Metric Sharpe-Optimal M2-Optimal
Expected Return         0.2721     0.2696
     Volatility         0.3066     0.3036
   Sharpe Ratio         2.1025     2.1025
             M²         0.2626     0.2626
